In [1]:
from pyspark.sql import SparkSession
 
# Crear una sesión de Spark
spark = SparkSession.builder \
    .appName("Scrap NYCTaxi") \
    .getOrCreate()

In [2]:
import json
from notebookutils import mssparkutils

# CONFIGURACIÓN
RUTA_JSON = f"abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/lista_urls.json"

# Limite
ANIO_LIMITE = 2026
MES_LIMITE = 2

# GENERADOR EXACTO HASTA FEBRERO 2026
tareas = []
print(f"Generando catálogo hasta {MES_LIMITE:02d}-{ANIO_LIMITE}...")

for anio in range(2009, ANIO_LIMITE + 1):
    
    # Si estamos en años anteriores, recorremos los 12 meses.
    # Si llegamos al año límite (2026), paramos en el mes límite (2).
    mes_final = 12 if anio < ANIO_LIMITE else MES_LIMITE
    
    for mes in range(1, mes_final + 1):
        for tipo in ["yellow", "green"]:
            
            # Regla de negocio: Taxis verdes empezaron en agosto de 2013
            if tipo == "green" and (anio < 2013 or (anio == 2013 and mes < 8)):
                continue
                
            archivo_parquet = f"{tipo}_tripdata_{anio}-{mes:02d}.parquet"
            tareas.append({
                "url_part": archivo_parquet,
                "anio": str(anio),
                "mes": f"{mes:02d}",
                "tipo": tipo
            })

# GUARDAR EL JSON EN EL DATA LAKE
json_texto = json.dumps(tareas, indent=2)
mssparkutils.fs.put(RUTA_JSON, json_texto, True)

print(f"¡Listo! Se ha generado el JSON con exactamente {len(tareas)} URLs.")
print("Ahora tu Pipeline se detendrá en febrero de 2026 y no creará carpetas basura.")

In [3]:
spark.stop()